In [3]:
import sys
import os
import numpy as np
import scipy.linalg as la
from scipy.signal import cont2discrete
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from mpl_toolkits.mplot3d import Axes3D 
import matplotlib as mpl

# Add project root to sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.append(project_root)

import utils.config as cfg
from utils.quadcopter_model import quad_hover_linear_model, create_discrete_model, gaussian_dist, gaussian_dist_time_varying, gaussian_wind_direction, visualize_gaussian_samples, visualize_multi_task_gaussians, visualize_multi_task_gaussians_3D, plot_disturbances_time, animate_trajectories, plot_angles, plot_xyz, plot_controls_grid, plot_angles_mean, plot_xyz_mean, plot_controls_grid_mean, summarize_checks
from utils.simple_MLP import PolicyMLP, rollout_policy, traj_cost, train_nn_policy, meta_train_nn_policy, maml_adapt, evaluate_policy_cost, plot_maml_diagnostics, plot_adapted_losses, plot_adaptation_comparison, plot_loss_evolution

# Plotting parameters
plt.style.use('default')
plt.rcParams['figure.dpi'] = 400

# Enable LaTeX rendering
mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"], 
    "axes.unicode_minus": False 
})

# Consistent printing of numpy arrays
np.set_printoptions(precision=3, suppress=True)

In [4]:
# Create and discretize model
A, B, C, D = quad_hover_linear_model()
# print("A =\n", A, "\n\nB =\n", B, "\n\nC =\n", C, "\n\nD =\n", D)

A_d, B_tilde_d, C_d, D_d = create_discrete_model(A, B, C, D)
B_d = B_tilde_d[:, :4]
# print("A_d =\n", A_d, "\n\nB_tilde_d =\n", B_tilde_d, "\n\nC_d =\n", C_d, "\n\nD_d =\n", D_d)

In [5]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyArrowPatch
from matplotlib.legend_handler import HandlerPatch

class HandlerArrow(HandlerPatch):
    def __init__(self, arrowstyle='->', mutation_scale=15, lw=2, **kwargs):
        super().__init__()
        self.arrowstyle = arrowstyle
        self.mutation_scale = mutation_scale
        self.lw = lw

    def create_artists(self, legend, orig_handle,
                       xdescent, ydescent, width, height, fontsize, trans):

        arrow = FancyArrowPatch(
            (0, height/2), (width, height/2),
            arrowstyle=self.arrowstyle,
            color=orig_handle.get_edgecolor(),
            mutation_scale=self.mutation_scale,
            lw=self.lw,
            joinstyle='miter',
            capstyle='butt',  
        )
        arrow.set_transform(trans)
        return [arrow]

def plot_wind_vectors(training_angles, training_means, testing_angle, testing_mean, filename):

    plt.figure(figsize=(5,5))

    # --- TRAINING VECTORS ---
    for ang, amp in zip(training_angles, training_means):
        x = amp * np.cos(np.deg2rad(ang))
        y = amp * np.sin(np.deg2rad(ang))
        plt.arrow(0, 0, x, y,
                  head_width=0.2, lw=1.5, length_includes_head=True,
                  color="blue", alpha=0.9)

    # --- TESTING VECTOR ---
    for ang, amp in zip(testing_angles, testing_means):
        x = amp * np.cos(np.deg2rad(ang))
        y = amp * np.sin(np.deg2rad(ang))
        plt.arrow(0, 0, x, y,
                  head_width=0.2, lw=1.5, length_includes_head=True,
                  color="red", alpha=0.9)
    
    # Mean training vector
    mean_amplitude = np.mean(training_means)
    mean_angle = np.mean(training_angles)
    plt.arrow(0, 0, mean_amplitude * np.cos(np.deg2rad(mean_angle)), mean_amplitude * np.sin(np.deg2rad(mean_angle)),
              head_width=0.2, lw=2.5, length_includes_head=True,
              color="purple", alpha=1)

    # Create dummy arrow handles for legend
    train_arrow = FancyArrowPatch((0,0), (1,0), color="blue")
    mean_train_arrow = FancyArrowPatch((0,0), (1,0), color="purple")
    test_arrow  = FancyArrowPatch((0,0), (1,0), color="red")


    plt.legend(
        [train_arrow, mean_train_arrow, test_arrow],
        ["Training wind", "Mean training wind", "Testing wind"],
        handler_map={
            FancyArrowPatch: HandlerArrow(
                arrowstyle='-|>',   # nicer arrow head
                mutation_scale=8,  # bigger arrowhead
                lw=1.5              # thicker line
            )
        },
        loc="upper left"
    )

    # Formatting
    plt.axhline(0, color="black", linewidth=0.8)
    plt.axvline(0, color="black", linewidth=0.8)
    plt.xlabel(r"$d_x$")
    plt.ylabel(r"$d_y$")
    plt.title("Wind Direction Vectors (mean amplitude)")
    plt.grid(True, alpha=0.3, linestyle='-.', zorder=0)
    plt.xlim(-6, 6)
    plt.ylim(-6, 6)
    plt.gca().set_aspect("equal", adjustable="box")

    plt.savefig(filename, dpi=300)
    plt.close()

In [6]:
np.random.seed(35)

# TRAINING TASKS: directions only in allowed quadrants
point_per_task = 50
num_tasks = 6

tasks_all_disturbances = []
tasks_all_x0 = []
tasks_dx = []
tasks_dy = []

training_angles = []  # to store angles for the vector plot
training_means  = []  # to store amplitudes for vector plot
training_sigmas = []  # to store sigmas for vector plot


print("=== TRAINING TASKS ===")
for i in range(num_tasks):

    # amplitude parameters
    mean  = 5.0
    sigma = 0.4

    # allowed wind direction quadrants
    direction_deg = np.random.uniform(0, 360)

    # generate disturbances
    dx, dy = gaussian_wind_direction(
        mean=mean,
        sigma=sigma,
        N=point_per_task,
        direction_deg=direction_deg,
        device="cpu"
    )

    dz = -cfg.G * torch.ones_like(dx)
    disturbances = torch.cat([dx, dy, dz], dim=-1)

    tasks_all_disturbances.append(disturbances)
    tasks_dx.append(dx)
    tasks_dy.append(dy)

    x0 = torch.zeros(point_per_task, 12)
    tasks_all_x0.append(x0)

    training_angles.append(direction_deg)
    training_means.append(mean)
    training_sigmas.append(sigma)

    print(f"[Task {i}] mean={mean:.2f}, sigma={sigma:.2f}, direction={direction_deg:.1f}°")

# Visualization of xyz distributions
visualize_multi_task_gaussians_3D(tasks_all_disturbances, filename="results_all_directions_extended/training_dist.png")

# Joint training dataset
tasks_all_disturbances_joint = torch.cat(tasks_all_disturbances, dim=0)
tasks_all_x0_joint = torch.cat(tasks_all_x0, dim=0).float()


# TESTING TASK: direction ONLY in quadrant III (180° to 270°)
print("\n=== TESTING TASK ===")
point_per_task = 50

disturbances_test = []
tasks_all_x0_test = []
tasks_dx_test = []
tasks_dy_test = []

testing_angles = []
testing_means  = []

wind_mean  = [5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0]
wind_sigma = [0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4]
wind_angle = [22.5, 67.5, 112.5, 157.5, 202.5, 247.5, 292.5, 337.5]

for i in range(len(wind_mean)):

    print(f"[Task {i}] mean={wind_mean[i]:.2f}, sigma={wind_sigma[i]:.2f}, direction={wind_angle[i]:.1f}°")

    dx_test, dy_test = gaussian_wind_direction(
        mean=wind_mean[i],
        sigma=wind_sigma[i],
        N=point_per_task,
        direction_deg=wind_angle[i],
        device="cpu"
    )

    dz_test = -cfg.G * torch.ones_like(dx_test)

    disturbances = torch.cat([dx_test, dy_test, dz_test], dim=-1)

    disturbances_test.append(disturbances)
    tasks_dx_test.append(dx_test)
    tasks_dy_test.append(dy_test)

    x0_test = torch.zeros(point_per_task, 12)
    tasks_all_x0_test.append(x0_test)

    testing_angles.append(wind_angle[i])
    testing_means.append(wind_mean[i])

visualize_multi_task_gaussians_3D(disturbances_test, filename="results_all_directions_extended/testing_dist.png")

plot_wind_vectors(training_angles, training_means, testing_angles, testing_means, "results_all_directions_extended/wind_vectors.png")

=== TRAINING TASKS ===
[Task 0] mean=5.00, sigma=0.40, direction=164.9°
[Task 1] mean=5.00, sigma=0.40, direction=217.0°
[Task 2] mean=5.00, sigma=0.40, direction=352.9°
[Task 3] mean=5.00, sigma=0.40, direction=266.1°
[Task 4] mean=5.00, sigma=0.40, direction=25.4°
[Task 5] mean=5.00, sigma=0.40, direction=198.0°

=== TESTING TASK ===
[Task 0] mean=5.00, sigma=0.40, direction=22.5°
[Task 1] mean=5.00, sigma=0.40, direction=67.5°
[Task 2] mean=5.00, sigma=0.40, direction=112.5°
[Task 3] mean=5.00, sigma=0.40, direction=157.5°
[Task 4] mean=5.00, sigma=0.40, direction=202.5°
[Task 5] mean=5.00, sigma=0.40, direction=247.5°
[Task 6] mean=5.00, sigma=0.40, direction=292.5°
[Task 7] mean=5.00, sigma=0.40, direction=337.5°


In [7]:
def angles_to_unit_vectors(angles_deg):
    angles_rad = np.deg2rad(angles_deg)
    return np.stack([np.cos(angles_rad), np.sin(angles_rad)], axis=-1)  # shape (N, 2)

def cosine_similarity_matrix(test_angles, train_angles):
    """
    Returns a matrix M where M[i,j] = cosine similarity between
    test direction i and training direction j.
    """
    test_vecs  = angles_to_unit_vectors(test_angles)     # shape (N_test, 2)
    train_vecs = angles_to_unit_vectors(train_angles)    # shape (N_train, 2)

    return test_vecs @ train_vecs.T                      # shape (N_test, N_train)

training_angles_np = np.array(training_angles)
testing_angles_np  = np.array(testing_angles)

similarity_matrix = cosine_similarity_matrix(testing_angles_np, training_angles_np)
mean_similarity = similarity_matrix.mean(axis=1)

print(similarity_matrix)
print("Mean similarity value for each test angle:", mean_similarity)
# -1 means very different directions (opposite), +1 means very similar directions

[[-0.792 -0.968  0.87  -0.445  0.999 -0.997]
 [-0.129 -0.862  0.266 -0.948  0.742 -0.649]
 [ 0.61  -0.251 -0.494 -0.896  0.051  0.079]
 [ 0.992  0.507 -0.964 -0.319 -0.67   0.761]
 [ 0.792  0.968 -0.87   0.445 -0.999  0.997]
 [ 0.129  0.862 -0.266  0.948 -0.742  0.649]
 [-0.61   0.251  0.494  0.896 -0.051 -0.079]
 [-0.992 -0.507  0.964  0.319  0.67  -0.761]]
Mean similarity value for each test angle: [-0.222 -0.263 -0.15   0.051  0.222  0.263  0.15  -0.051]


In [13]:
mean_angle = np.mean(training_angles)
print(mean_angle)
testing_angles_np  = np.array(testing_angles)

similarity_values =  []

for angles in testing_angles_np:
    similarity = np.cos(np.deg2rad(angles -  mean_angle))
    print(angles -  mean_angle)
    similarity_values.append(similarity)

similarity_values =  np.array(similarity_values)
print(similarity_values)


204.05146134178378
-181.55146134178378
-136.55146134178378
-91.55146134178378
-46.55146134178378
-1.551461341783778
43.44853865821622
88.44853865821622
133.44853865821622
[-1.    -0.726 -0.027  0.688  1.     0.726  0.027 -0.688]


In [ ]:
# Previous 

=== TRAINING TASKS ===
[Task 0] mean=1.33, sigma=0.54, direction=74.4°
[Task 1] mean=5.87, sigma=0.27, direction=134.0°
[Task 2] mean=1.43, sigma=0.46, direction=286.6°
[Task 3] mean=0.71, sigma=0.13, direction=321.8°
[Task 4] mean=2.95, sigma=0.29, direction=273.7°
[Task 5] mean=5.15, sigma=0.36, direction=64.8°

=== TESTING TASK ===
[Testing task] mean=5.92, sigma=0.32, direction=259.9°

-0.05116666666666669


In [10]:
# JOINT TRAINING (BASELINE)
device = "cpu"
epochs = 6000
compute_mean_every = 100
lr = 1e-3

joint_policy, train_means = train_nn_policy(
    A_d_nt=A_d, B_tilde_d_nt=B_tilde_d,
    init_points=tasks_all_x0_joint, disturbances=tasks_all_disturbances_joint,
    lr=lr, epochs=epochs,
    device=device, verbose_every=compute_mean_every, shuffle_each_epoch=True
)

[ep 100/6000]:   train_loss=1.7258e+05   |    eval_loss=1.7232e+05   |   |x_T|_avg=94.405
[ep 200/6000]:   train_loss=1.1171e+05   |    eval_loss=1.1352e+05   |   |x_T|_avg=69.970
[ep 300/6000]:   train_loss=6.3593e+04   |    eval_loss=6.2514e+04   |   |x_T|_avg=58.971
[ep 400/6000]:   train_loss=3.2791e+04   |    eval_loss=2.9435e+04   |   |x_T|_avg=25.091
[ep 500/6000]:   train_loss=2.4304e+04   |    eval_loss=2.3247e+04   |   |x_T|_avg=30.593
[ep 600/6000]:   train_loss=1.6822e+04   |    eval_loss=1.6676e+04   |   |x_T|_avg=25.177
[ep 700/6000]:   train_loss=1.2724e+04   |    eval_loss=1.2370e+04   |   |x_T|_avg=16.947
[ep 800/6000]:   train_loss=9.3154e+03   |    eval_loss=9.2106e+03   |   |x_T|_avg=17.001
[ep 900/6000]:   train_loss=5.6845e+03   |    eval_loss=5.7712e+03   |   |x_T|_avg=10.794
[ep 1000/6000]:   train_loss=1.4895e+03   |    eval_loss=1.4637e+03   |   |x_T|_avg=3.288
[ep 1100/6000]:   train_loss=5.8306e+02   |    eval_loss=5.8719e+02   |   |x_T|_avg=1.347
[ep 1200/6

In [12]:
def plot_loss_evolution(train_means, total_epochs, log_interval, filename="training_loss_evolution.png"):
    plt.figure(figsize=(7, 5))

    epochs_range = np.arange(log_interval, total_epochs + 1, log_interval)

    plt.plot(epochs_range, train_means, 'blue', alpha=0.8)
    plt.yscale("log")
    plt.xlabel("Epoch")
    plt.ylabel("Mean training loss")
    plt.grid(True, which="both", ls="--", lw=0.4)
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()
plot_loss_evolution(train_means, epochs, compute_mean_every, filename="results_all_directions_extended/loss_joint_training.png")

In [6]:
# META TRAINING
device = "cpu"
epochs = 6000
compute_mean_every = 100
lr_outer_loop = 1e-3            # β
lr_inner_loop = 1e-3            # α
nb_samples_inner_training = 5
nb_samples_inner_testing = 5


meta_policy, logs = meta_train_nn_policy(
    A_d_nt = A_d, B_tilde_d_nt = B_tilde_d,
    init_points_list = tasks_all_x0,          
    disturbances_list = tasks_all_disturbances,
    lr_outer=lr_outer_loop,          
    lr_inner=lr_inner_loop,   
    meta_batch_size=None, 
    K_inner=nb_samples_inner_training,               
    K_outer=nb_samples_inner_testing,    
    epochs=epochs,
    device="cpu",
    verbose_every=compute_mean_every
)

[MAML] Meta-training on 6 tasks.


/Users/mbacyril/Documents/GitHub/MetaLearning-Control-Quadcopters/utils/simple_MLP.py:373: FutureWarning: `torch.nn.utils.stateless.functional_call` is deprecated as of PyTorch 2.0 and will be removed in a future version of PyTorch. Please use `torch.func.functional_call` instead which is a drop-in replacement.
  return functional_call(theta_policy, theta_i_prime, (x,))


[ep 100/6000] meta_loss = 4.4424e+05
[ep 200/6000] meta_loss = 4.2384e+05
[ep 300/6000] meta_loss = 2.7714e+05
[ep 400/6000] meta_loss = 2.0146e+05
[ep 500/6000] meta_loss = 1.4933e+05
[ep 600/6000] meta_loss = 1.6895e+05
[ep 700/6000] meta_loss = 1.4052e+05
[ep 800/6000] meta_loss = 1.0726e+05
[ep 900/6000] meta_loss = 9.8027e+04
[ep 1000/6000] meta_loss = 8.6921e+04
[ep 1100/6000] meta_loss = 6.5211e+04
[ep 1200/6000] meta_loss = 5.6363e+04
[ep 1300/6000] meta_loss = 4.6942e+04
[ep 1400/6000] meta_loss = 3.8227e+04
[ep 1500/6000] meta_loss = 3.2174e+04
[ep 1600/6000] meta_loss = 2.6425e+04
[ep 1700/6000] meta_loss = 2.5428e+04
[ep 1800/6000] meta_loss = 2.3628e+04
[ep 1900/6000] meta_loss = 2.8496e+04
[ep 2000/6000] meta_loss = 2.1162e+04
[ep 2100/6000] meta_loss = 1.8182e+04
[ep 2200/6000] meta_loss = 1.8600e+04
[ep 2300/6000] meta_loss = 1.6199e+04
[ep 2400/6000] meta_loss = 1.4146e+04
[ep 2500/6000] meta_loss = 1.3716e+04
[ep 2600/6000] meta_loss = 1.3653e+04
[ep 2700/6000] meta_l

In [7]:
def plot_maml_diagnostics(logs, filename):
    epochs = np.array(logs["epochs_logged"])

    # META LOSS
    plt.figure(figsize=(7,5))
    plt.plot(epochs, logs["meta_loss"], 'blue', alpha=0.8, label="Meta-loss")
    plt.yscale("log")
    plt.xlabel("Epoch")
    plt.ylabel("Meta loss")
    plt.grid(True, ls="--", lw=0.4)
    plt.tight_layout()
    plt.savefig(filename.replace(".png", "_outer.png"))
    plt.close()

    # PER-TASK INNER LOSSES
    inner_per_task = np.array(logs["inner_loss_tasks"])   # shape [num_logs, num_tasks]
    num_tasks = inner_per_task.shape[1]

    plt.figure(figsize=(7,5))
    for t in range(num_tasks):
        plt.plot(epochs, inner_per_task[:, t], 
        label=fr"Task ($\mathcal{{N}}({training_means[t]:.1f}, {training_sigmas[t]:.1f}),\; {training_angles[t]:.0f}^\circ$)")
    plt.yscale("log")
    plt.xlabel("Epoch")
    plt.ylabel("Inner losses")
    plt.legend()
    plt.grid(True, ls="--", lw=0.4)
    plt.tight_layout()
    plt.savefig(filename.replace(".png", "_inner.png"))
    plt.close()

plot_maml_diagnostics(logs, filename="results_all_directions_extended/loss_meta_training.png")

In [46]:
from torch.nn.utils.stateless import functional_call

# Weights adaptation on a new unseen task
def maml_adapt(theta_policy, A_d_nt, B_tilde_d_nt, x0_test, disturbances_test, lr_inner, nb_gradient_descent=5, K_samples=5, device="cpu"):
    """
    Returns:
        theta_adapt : dict of adapted parameters
        adapt_losses : inner-loop losses (K samples)
        full_mean_losses : mean loss over ALL test samples at each step
        full_min_losses : min loss over ALL test samples
        full_max_losses : max loss over ALL test samples
    """

    # Convert dynamics
    A_d = torch.tensor(A_d_nt, dtype=torch.float32, device=device)
    B_tilde_d = torch.tensor(B_tilde_d_nt, dtype=torch.float32, device=device)

    # Prepare full test dataset
    x0_full = x0_test.to(device)
    dist_full = disturbances_test.to(device)

    # Sample K points for adaptation
    N = x0_test.shape[0]
    perm = torch.randperm(N)[:K_samples]
    x0_adapt  = x0_full[perm]
    dist_adapt = dist_full[perm]

    # Copy meta-parameters θ → θ_adapt
    theta_adapt = {name: p.clone() for name, p in theta_policy.named_parameters()}

    adapt_losses = []
    full_mean_losses = []
    full_min_losses  = []
    full_max_losses  = []

    # Helper forward function
    def forward_with_params(x):
        return functional_call(theta_policy, theta_adapt, (x,))

    # Begin adaptation loop
    for step in range(nb_gradient_descent):

        # FULL EVALUATION (NO GRAD) on ALL TESTS
        with torch.no_grad():
            Xf, Uf = rollout_policy(A_d, B_tilde_d, forward_with_params,
                                    x0_full, dist_full)
            # Loss per trajectory
            traj_losses = traj_cost(Xf, Uf)       # shape [N]
            full_mean_losses.append(traj_losses.mean().item())
            full_min_losses.append(traj_losses.min().item())
            full_max_losses.append(traj_losses.max().item())

        # ---- INNER-LOOP LOSS on K samples ----
        X, U = rollout_policy(A_d, B_tilde_d, forward_with_params,
                              x0_adapt, dist_adapt)
        inner_loss = traj_cost(X, U).mean()
        adapt_losses.append(inner_loss.item())

        # Compute gradients
        grads = torch.autograd.grad(inner_loss,
                                    tuple(theta_adapt.values()),
                                    create_graph=False)

        # Gradient clipping
        max_norm = 1.0
        total_norm = torch.sqrt(sum(g.norm()**2 for g in grads))
        clip_coef = max_norm / (total_norm + 1e-6)
        if clip_coef < 1:
            grads = [g * clip_coef for g in grads]

        # Update parameters
        theta_adapt = {
            n: p - lr_inner * g
            for (n, p), g in zip(theta_adapt.items(), grads)
        }

        print(f"[Adapt {step+1}] inner_loss={inner_loss.item():.4f}, "
             f"full_mean={full_mean_losses[-1]:.4f}")

    return theta_adapt, adapt_losses, full_mean_losses, full_min_losses, full_max_losses

In [47]:
# ADAPTATION ON TEST TASK WITH META-LEARNED POLICY
all_full_adapt_loss_meta = []

# Set seed for reproducibility
torch.manual_seed(108)

for i in range(len(wind_mean)):
    adapted_policy_from_meta_learning, adapt_losses_meta, full_mean_losses_meta, full_min_losses_meta, full_max_losses_meta = maml_adapt(
        meta_policy,
        A_d, B_tilde_d,
        x0_test, disturbances_test[i],
        lr_inner=2e-2,
        nb_gradient_descent=6,
        K_samples=5,
        device="cpu"
    )
    all_full_adapt_loss_meta.append(full_mean_losses_meta)
    print(f"Completed adaptation for testing task {i}\n")
    

def plot_adaptation_comparison_test(all_full_adapt_loss_meta):
    steps = np.arange(0, 6)
    plt.figure(figsize=(7,4.25))

    colors = plt.cm.Dark2(np.linspace(0, 1, len(all_full_adapt_loss_meta)))

    for i in range(len(all_full_adapt_loss_meta)):
        full_mean_losses_meta = all_full_adapt_loss_meta[i]

        means =  [5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0]
        sigmas = [0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4]
        angles = [0, 45, 90, 135, 180, 225, 270, 315]

        plt.plot(steps, full_mean_losses_meta, marker='o', markersize=4, linestyle='-', 
                label=fr"Meta adapt ($\mathcal{{N}}({means[i]:.1f}, {sigmas[i]:.1f}),\; {angles[i]:.0f}^\circ$)", color=colors[i], alpha=0.9)

    plt.yscale("log")
    plt.xlabel("Adaptation Step")
    plt.ylabel("Full Test Loss")
    plt.xlim(0, 5)
    plt.grid(True)

    plt.legend(
        bbox_to_anchor=(1.02, 0.5),
        loc="center left",
        ncol=1,
        frameon=True,          # <-- enables the box
        fancybox=True,         # rounded corners (optional)
        shadow=False,          # you can enable if you want a shadow
        borderpad=0.8          # spacing inside the box
    )
    plt.subplots_adjust(right=0.78)

    plt.tight_layout()
    plt.savefig("results_all_directions_extended/just_meta_adapt.png", dpi=300)
    plt.close()

plot_adaptation_comparison_test(all_full_adapt_loss_meta)

/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_98756/335470091.py:38: FutureWarning: `torch.nn.utils.stateless.functional_call` is deprecated as of PyTorch 2.0 and will be removed in a future version of PyTorch. Please use `torch.func.functional_call` instead which is a drop-in replacement.
  return functional_call(theta_policy, theta_adapt, (x,))


[Adapt 1] inner_loss=294.7306, full_mean=298.1978
[Adapt 2] inner_loss=281.3762, full_mean=529.7076
[Adapt 3] inner_loss=292.4398, full_mean=301.6499
[Adapt 4] inner_loss=277.0062, full_mean=282.5827
[Adapt 5] inner_loss=272.4935, full_mean=280.1967
[Adapt 6] inner_loss=267.2082, full_mean=275.7390
Completed adaptation for testing task 0

[Adapt 1] inner_loss=232.5979, full_mean=242.4899
[Adapt 2] inner_loss=233.0513, full_mean=278.9731
[Adapt 3] inner_loss=224.8758, full_mean=235.4030
[Adapt 4] inner_loss=213.7111, full_mean=222.0896
[Adapt 5] inner_loss=218.3041, full_mean=229.5965
[Adapt 6] inner_loss=207.1533, full_mean=215.1315
Completed adaptation for testing task 1

[Adapt 1] inner_loss=982459.3750, full_mean=945104.8125
[Adapt 2] inner_loss=18780.7871, full_mean=16750.3242
[Adapt 3] inner_loss=398.9697, full_mean=410.4180
[Adapt 4] inner_loss=265.5532, full_mean=263.8879
[Adapt 5] inner_loss=253.4362, full_mean=262.0078
[Adapt 6] inner_loss=254.0358, full_mean=252.5205
Complete

In [73]:
# APAPTATION ON TEST TASK WITH JOINTLY TRAINED POLICY
all_full_adapt_loss_joint = []

# Set seed for reproducibility # 19 146 379
n = np.random.randint(200)
print(n)
torch.manual_seed(n)

for i in range(len(wind_mean)):

    adapted_policy_from_joint_learning, adapt_losses_joint, full_mean_losses_joint, full_min_losses_joint, full_max_losses_joint = maml_adapt(
        joint_policy,
        A_d, B_tilde_d,
        x0_test, disturbances_test[i],
        lr_inner=5e-3,
        nb_gradient_descent=6,
        K_samples=5,
        device="cpu"
    )
    all_full_adapt_loss_joint.append(full_mean_losses_joint)
    print(f"Completed adaptation for testing task {i}\n")


def plot_adaptation_comparison_test(all_full_adapt_loss_meta):
    steps = np.arange(0, 6)
    plt.figure(figsize=(7,4.25))

    colors = plt.cm.Dark2(np.linspace(0, 1, len(all_full_adapt_loss_meta)))

    for i in range(len(all_full_adapt_loss_meta)):
        full_mean_losses_meta = all_full_adapt_loss_meta[i]

        means =  [5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0]
        sigmas = [0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4]
        angles = [0, 45, 90, 135, 180, 225, 270, 315]

        plt.plot(steps, full_mean_losses_meta, marker='o', markersize=4, linestyle='-.', 
                label=fr"Joint adapt ($\mathcal{{N}}({means[i]:.1f}, {sigmas[i]:.1f}),\; {angles[i]:.0f}^\circ$)", color=colors[i], alpha=0.9)

    plt.yscale("log")
    plt.xlabel("Adaptation Step")
    plt.ylabel("Full Test Loss")
    plt.xlim(0, 5)
    plt.grid(True)

    plt.legend(
        bbox_to_anchor=(1.02, 0.5),
        loc="center left",
        ncol=1,
        frameon=True,          # <-- enables the box
        fancybox=True,         # rounded corners (optional)
        shadow=False,          # you can enable if you want a shadow
        borderpad=0.8          # spacing inside the box
    )
    plt.subplots_adjust(right=0.78)

    plt.tight_layout()
    plt.savefig("results_all_directions_extended/just_joint_adapt.png", dpi=300)
    plt.close()

plot_adaptation_comparison_test(all_full_adapt_loss_joint)

123
[Adapt 1] inner_loss=268.7706, full_mean=281.3549
[Adapt 2] inner_loss=257.3333, full_mean=298.1862


/var/folders/81/fnt9x6_x5l343vfsqmvf5fd40000gn/T/ipykernel_98756/335470091.py:38: FutureWarning: `torch.nn.utils.stateless.functional_call` is deprecated as of PyTorch 2.0 and will be removed in a future version of PyTorch. Please use `torch.func.functional_call` instead which is a drop-in replacement.
  return functional_call(theta_policy, theta_adapt, (x,))


[Adapt 3] inner_loss=249.3509, full_mean=421.6683
[Adapt 4] inner_loss=241.5867, full_mean=468.8090
[Adapt 5] inner_loss=233.4162, full_mean=317.8566
[Adapt 6] inner_loss=223.6826, full_mean=232.7699
Completed adaptation for testing task 0

[Adapt 1] inner_loss=276.4574, full_mean=280.2848
[Adapt 2] inner_loss=246.9893, full_mean=250.7356
[Adapt 3] inner_loss=237.1129, full_mean=879.8242
[Adapt 4] inner_loss=231.3999, full_mean=418.3210
[Adapt 5] inner_loss=226.7419, full_mean=244.8252
[Adapt 6] inner_loss=222.5162, full_mean=221.5903
Completed adaptation for testing task 1

[Adapt 1] inner_loss=82519.4609, full_mean=79296.1562
[Adapt 2] inner_loss=60681.9570, full_mean=49928.1055
[Adapt 3] inner_loss=56491.6484, full_mean=50890.5039
[Adapt 4] inner_loss=50743.0938, full_mean=64709.1016
[Adapt 5] inner_loss=22262.0996, full_mean=32555.0957
[Adapt 6] inner_loss=26847.0117, full_mean=45632.6719
Completed adaptation for testing task 2

[Adapt 1] inner_loss=474.4863, full_mean=317.7470
[Ad

In [75]:
def plot_adaptation_comparison_all(all_full_adapt_loss_meta, all_full_adapt_loss_joint, filename):
    steps = np.arange(0, 6)
    plt.figure(figsize=(8,4.5))

    colors = plt.cm.Dark2(np.linspace(0, 1, len(all_full_adapt_loss_meta)))
    color_joint = 'black' # "#2E7D32" 
    color_meta = 'black'  # "#D81B60"

    for i in range(len(all_full_adapt_loss_meta)):
        full_mean_losses_meta = all_full_adapt_loss_meta[i]
        full_mean_losses_joint = all_full_adapt_loss_joint[i]

        means =  [5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0]
        sigmas = [0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4]
        angles = [0,   45,  90, 135, 180, 225, 270, 315]

        plt.plot(
            steps, full_mean_losses_meta,
            linewidth=1.5, linestyle='-',
            label = (fr"Meta adapt ($\mathcal{{N}}({wind_mean[i]:.1f}, {wind_sigma[i]:.1f}),\; {wind_angle[i]:.0f}^\circ$), $\bar{{S}}_{{cos}} = {mean_similarity[i]:.3f}$"),
            color=colors[i], alpha=0.6
        )

        plt.plot(
            steps, full_mean_losses_joint,
            linewidth=1.5, linestyle='-.',
            label = (fr"Joint adapt ($\mathcal{{N}}({wind_mean[i]:.1f}, {wind_sigma[i]:.1f}),\; {wind_angle[i]:.0f}^\circ$), $\bar{{S}}_{{cos}} = {mean_similarity[i]:.3f}$"),
            color=colors[i], alpha=0.6
        )

    mean_meta = np.mean(np.array(all_full_adapt_loss_meta), axis=0)
    mean_joint = np.mean(np.array(all_full_adapt_loss_joint), axis=0)

    plt.plot(
        steps, mean_meta,
        linewidth=2, linestyle='-',
        label="Mean meta adaptation",
        color=color_meta
    )

    plt.plot(
        steps, mean_joint,
        linewidth=2, linestyle='-.', 
        label="Mean joint adaptation",
        color=color_joint
    )

    plt.yscale("log")
    plt.xlabel("Adaptation Step")
    plt.ylabel("Full Test Loss")
    plt.xlim(0, 5)
    plt.grid(True)

    plt.legend(
        bbox_to_anchor=(1.02, 0.5),
        loc="center left",
        ncol=1,
        frameon=True,
        fancybox=True,
        borderpad=0.8
    )
    plt.subplots_adjust(right=0.78)

    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()

plot_adaptation_comparison_all(all_full_adapt_loss_meta, all_full_adapt_loss_joint, "results_all_directions_extended/adaptation_comparison.png")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


def plot_adaptation_comparison_all(all_full_adapt_loss_meta,
                                   all_full_adapt_loss_joint,
                                   filename):

    steps = np.arange(0, 6)
    plt.figure(figsize=(8, 4.5))

    colors = plt.cm.Dark2(np.linspace(0, 1, len(all_full_adapt_loss_meta)))

    color_joint = 'black'
    color_meta  = 'black'

    # Fixed means/sigmas/angles as indexed
    means  = [5.0] * 8
    sigmas = [0.4] * 8
    angles = [0, 45, 90, 135, 180, 225, 270, 315]

    # --- Plot curves for each direction ---
    for i in range(len(all_full_adapt_loss_meta)):
        full_mean_losses_meta  = all_full_adapt_loss_meta[i]
        full_mean_losses_joint = all_full_adapt_loss_joint[i]

        # Meta
        plt.plot(
            steps, full_mean_losses_meta,
            linewidth=1.5, linestyle='-',
            color=colors[i], alpha=0.6,
        )

        # Joint
        plt.plot(
            steps, full_mean_losses_joint,
            linewidth=1.5, linestyle='-.',
            color=colors[i], alpha=0.6,
        )

    # --- Mean curves ---
    mean_meta  = np.mean(np.array(all_full_adapt_loss_meta), axis=0)
    mean_joint = np.mean(np.array(all_full_adapt_loss_joint), axis=0)

    plt.plot(
        steps, mean_meta,
        linewidth=2, linestyle='-',
        color=color_meta, label="Mean meta adaptation"
    )

    plt.plot(
        steps, mean_joint,
        linewidth=2, linestyle='-.',
        color=color_joint, label="Mean joint adaptation"
    )

    # ============================
    # CUSTOM LEGEND WITH COLORS
    # ============================
    legend_handles = []

    for i in range(len(all_full_adapt_loss_meta)):
        # Meta legend entry
        legend_handles.append(Line2D(
            [0], [0],
            color=colors[i], linestyle='-', linewidth=1.5,
            label=(fr"Meta adapt ($\mathcal{{N}}({means[i]:.1f}, {sigmas[i]:.1f}),\; "
                   fr"{angles[i]:.0f}^\circ$), $\bar{{S}}_{{cos}} = {mean_similarity[i]:.3f}$")
        ))

        # Joint legend entry
        legend_handles.append(Line2D(
            [0], [0],
            color=colors[i], linestyle='-.', linewidth=1.5,
            label=(fr"Joint adapt ($\mathcal{{N}}({means[i]:.1f}, {sigmas[i]:.1f}),\; "
                   fr"{angles[i]:.0f}^\circ$), $\bar{{S}}_{{cos}} = {mean_similarity[i]:.3f}$")
        ))

    # Add mean curves (black)
    legend_handles.append(Line2D(
        [0], [0],
        color='black', linestyle='-', linewidth=2,
        label="Mean meta adaptation"
    ))

    legend_handles.append(Line2D(
        [0], [0],
        color='black', linestyle='-.', linewidth=2,
        label="Mean joint adaptation"
    ))

    # Render custom legend
    legend = plt.legend(
        handles=legend_handles,
        bbox_to_anchor=(1.02, 0.5),
        loc="center left",
        ncol=1,
        frameon=True,
        fancybox=True,
        borderpad=0.8
    )

    # -------- FIX: color the legend TEXT with curve colors --------
    legend_texts = legend.get_texts()
    idx_text = 0

    for i in range(len(all_full_adapt_loss_meta)):
        # meta text
        legend_texts[idx_text].set_color(colors[i])
        idx_text += 1

        # joint text
        legend_texts[idx_text].set_color(colors[i])
        idx_text += 1

    # =======================
    # Plot formatting
    # =======================
    plt.yscale("log")
    plt.xlabel("Adaptation Step")
    plt.ylabel("Full Test Loss")
    plt.xlim(0, 5)
    plt.grid(True)

    plt.subplots_adjust(right=0.78)
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()

plot_adaptation_comparison_all(all_full_adapt_loss_meta, all_full_adapt_loss_joint, "results_all_directions_extended/adaptation_comparison.png")

In [ ]:
# TESTING ADAPTED POLICY FROM META-LEARNING
from torch.nn.utils.stateless import functional_call

def forward_with_params(x):
    return functional_call(meta_policy, adapted_policy_from_meta_learning, (x,))
    # take the model architecture of theta_policy, replace its parameters by theta_adapt

A_d = torch.tensor(A_d, dtype=torch.float32, device=device)
B_tilde_d = torch.tensor(B_tilde_d, dtype=torch.float32, device=device)
X_test_adapted_from_meta, U_test_adapted_from_meta = rollout_policy(
    A_d, B_tilde_d, forward_with_params,
    x0_test.to(device), disturbances_test.to(device)
)
costs = traj_cost(X_test_adapted_from_meta, U_test_adapted_from_meta)
print("Adapted test cost =", costs.mean().item())

plot_xyz_mean(X_test_adapted_from_meta, filename="results_all_directions_extended/testing_xyz_adapted_from_meta.png")
plot_angles_mean(X_test_adapted_from_meta, filename="results_all_directions_extended/testing_angles_adapted_from_meta.png")
plot_controls_grid_mean(U_test_adapted_from_meta, filename="results_all_directions_extended/testing_controls_adapted_from_meta.png")
animate_trajectories(X_test_adapted_from_meta, "results_all_directions_extended/testing_xyz_3d_adapted_from_meta.mp4")

In [ ]:
# TESTING ADAPTED POLICY FROM JOINT LEARNING
from torch.nn.utils.stateless import functional_call

def forward_with_params(x):
    return functional_call(joint_policy, adapted_policy_from_joint_learning, (x,))
    # take the model architecture of theta_policy, replace its parameters by theta_adapt

A_d = torch.tensor(A_d, dtype=torch.float32, device=device)
B_tilde_d = torch.tensor(B_tilde_d, dtype=torch.float32, device=device)
X_test_adapted_from_joint, U_test_adapted_from_joint = rollout_policy(
    A_d, B_tilde_d, forward_with_params,
    x0_test.to(device), disturbances_test.to(device)
)
costs = traj_cost(X_test_adapted_from_joint, U_test_adapted_from_joint)
print("Adapted test cost =", costs.mean().item())

plot_xyz_mean(X_test_adapted_from_joint, filename="results_all_directions_extended/testing_xyz_adapted_from_joint.png")
plot_angles_mean(X_test_adapted_from_joint, filename="results_all_directions_extended/testing_angles_adapted_from_joint.png")
plot_controls_grid_mean(U_test_adapted_from_joint, filename="results_all_directions_extended/testing_controls_adapted_from_joint.png")
animate_trajectories(X_test_adapted_from_joint, "results_all_directions_extended/testing_xyz_3d_adapted_from_joint.mp4")

In [ ]:
# Compute improvement percentage plot
meta = np.array(full_mean_losses_meta)
joint = np.array(full_mean_losses_joint)

# Percentage improvement of meta vs joint
improvement_pct = 100 * (joint - meta) / joint

steps = np.arange(0, len(improvement_pct))

plt.figure(figsize=(6,4))
plt.plot(steps, improvement_pct, marker='o', markersize=4, color='purple', alpha=0.8)

plt.xlabel("Adaptation Step")
plt.ylabel("Loss improvement (\%)")
plt.ylim(0, 100)
plt.xlim(0, 5)
plt.title("Improvement of Meta-Learning vs Joint Training")
plt.grid(True)
plt.tight_layout()
plt.savefig("results_all_directions_extended/improvement_percentage.png", dpi=300)
plt.close()

In [ ]:
# Robustness: adaptation with different 5 samples from test distribution

all_robust_full_mean_losses_joint = []
all_robust_full_mean_losses_meta = []

for i in range(10):
    print(f"\n=== ROBUSTNESS TEST {i+1} ===")

    # APAPTATION ON TEST TASK WITH JOINTLY TRAINED POLICY
    robust_adapted_policy_from_joint_learning, robust_adapt_losses_joint, robust_full_mean_losses_joint, robust_full_min_losses_joint, robust_full_max_losses_joint = maml_adapt(
        joint_policy,
        A_d, B_tilde_d,
        x0_test, disturbances_test,
        lr_inner=1e-2,
        nb_gradient_descent=6,
        K_samples=5,
        device="cpu"
    )
    all_robust_full_mean_losses_joint.append(robust_full_mean_losses_joint)


    # ADAPTATION ON TEST TASK WITH META-LEARNED POLICY
    robust_adapted_policy_from_meta_learning, robust_adapt_losses_meta, robust_full_mean_losses_meta, robust_full_min_losses_meta, robust_full_max_losses_meta = maml_adapt(
        meta_policy,
        A_d, B_tilde_d,
        x0_test, disturbances_test,
        lr_inner=5e-2,
        nb_gradient_descent=6,
        K_samples=5,
        device="cpu"
    )
    all_robust_full_mean_losses_meta.append(robust_full_mean_losses_meta)

In [ ]:
# Plot all robustness curves
plt.figure(figsize=(8, 5))

color_joint = "#2E7D32"   # forest green
color_meta  = "#D81B60"   # magenta

# JOINT TRAINING CURVES (5 curves)
for i, losses in enumerate(all_robust_full_mean_losses_joint):
    plt.plot(
        losses,
        color=color_joint,
        alpha=0.6,
        linewidth=2,
        label="Joint adaptation with 5 samples from test distribution" if i == 0 else None
    )

# META-LEARNING CURVES (5 curves)
for i, losses in enumerate(all_robust_full_mean_losses_meta):
    plt.plot(
        losses,
        color=color_meta,
        alpha=0.6,
        linewidth=2,
        label="Meta adaptation with 5 samples from test distribution" if i == 0 else None
    )

plt.xlabel("Adaptation step")
plt.ylabel("Mean full test loss")
plt.yscale("log")
plt.xlim(0, 5)
plt.grid(True, ls="--", lw=0.4)
plt.legend()
plt.tight_layout()
plt.savefig("results_all_directions_extended/robustness_adaptation_comparison.png", dpi=300)
plt.close()